In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
from pathlib import Path
from dotenv import load_dotenv

# Jupyter often runs with cwd at the repo root; load_dotenv() would otherwise
# pick up ~/.env from parent directories instead of this course's .env.
_env = Path.cwd() / ".env"
if not _env.is_file():
    _env = Path.cwd() / "anthropic-partner-course" / ".env"
load_dotenv(_env)

True

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text.strip())

In [7]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL in the format 's3://bucket-name.region.amazonaws.com'. Return the region name only."}, {'task': "Create a JSON object representing an AWS IAM policy that grants read-only access to a specific S3 bucket named 'my-data-bucket'."}, {'task': "Write a regular expression that matches valid AWS EC2 instance IDs, which follow the format 'i-' followed by 17 hexadecimal characters (e.g., i-0742d86d9e7212345)."}]


In [8]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [9]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [10]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [11]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [12]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [13]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_region_from_s3_url(url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Expected format: s3://bucket-name.region.amazonaws.com\n    \n    Args:\n        url (str): The S3 bucket URL\n        \n    Returns:\n        str: The region name (e.g., 'us-east-1')\n        \n    Raises:\n        ValueError: If the URL format is invalid or region cannot be extracted\n    \"\"\"\n    # Pattern to match s3://bucket-name.region.amazonaws.com\n    pattern = r'^s3://[a-z0-9.-]+\\.([a-z0-9-]+)\\.amazonaws\\.com/?$'\n    \n    match = re.match(pattern, url)\n    \n    if not match:\n        raise ValueError(f\"Invalid S3 URL format: {url}\")\n    \n    region = match.group(1)\n    return region\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Test cases\n    test_urls = [\n        \"s3://my-buc